<a href="https://colab.research.google.com/github/pushkar-hue/LLM/blob/main/gemma14M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-03 05:50:27--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-07-03 05:50:28 (35.4 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [5]:
with open('input.txt', 'r') as f:
  text = f.read()

len(text)

1115394

In [6]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [7]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [8]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [9]:
encode('hello, world')

[46, 43, 50, 50, 53, 6, 1, 61, 53, 56, 50, 42]

In [10]:
decode(encode('hello, world'))

'hello, world'

In [11]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [12]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [13]:
train_data.shape, val_data.shape

(torch.Size([1003854]), torch.Size([111540]))

In [14]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [15]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"when input is {context} the target is {target}")

when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [16]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [17]:
torch.manual_seed(1337)

batch_size = 4
block_size = 8

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data)-block_size,   (batch_size,))
  x = torch.stack([data[i: i+block_size] for i in ix])
  y = torch.stack([data[i + 1: i+block_size+1] for i in ix])
  return x.to(device), y.to(device)

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
print('--------')

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b, :t+1]
    target = yb[b, t]
    print(f'when input is {context} the output is {target}.')

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
--------
when input is tensor([24]) the output is 43.
when input is tensor([24, 43]) the output is 58.
when input is tensor([24, 43, 58]) the output is 5.
when input is tensor([24, 43, 58,  5]) the output is 57.
when input is tensor([24, 43, 58,  5, 57]) the output is 1.
when input is tensor([24, 43, 58,  5, 57,  1]) the output is 46.
when input is tensor([24, 43, 58,  5, 57,  1, 46]) the output is 43.
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the output is 39.
when input is tensor([44]) the output is 53.
when input is tensor([44, 53]) the output is 56.
when input is tensor([44, 53, 56])

In [18]:
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
head_size = n_embd // n_head

In [19]:
import torch
import torch.nn as nn
from torch.nn import functional as F


In [27]:
import torch
block_size=8
mask = torch.tril(torch.ones(block_size, block_size))

mask = torch.triu(mask, diagonal=-2)
mask

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [0., 1., 1., 1., 0., 0., 0., 0.],
        [0., 0., 1., 1., 1., 0., 0., 0.],
        [0., 0., 0., 1., 1., 1., 0., 0.],
        [0., 0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 0., 0., 1., 1., 1.]])

In [ ]:
class Head(nn.Module):

  def __init__(self, head_size, window_size=None):
    super().__init__()

    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)

    mask =  torch.tril(torch.ones(block_size, block_size))

    if window_size is not None:
      mask = torch.triu(mask, diagonal=-window_size)

    self.register_buffer('tril', mask)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B, T, C = x.shape

    k = self.key(x)
    q = self.query(x) # Changed q = self.query(q) to q = self.query(x)
    v = self.value(x) # Changed v = self.value(v) to v = self.value(x)

    wei = q @ k.transpose(-2, -1) * head_size**-0.5
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)

    out = wei @ v
    return out

In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embd, n_embd, bias=False)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.proj(out)
    out = self.dropout(out)
    return out

In [ ]:
class GemmaFeedForward(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.gate = nn.Linear(n_embd, 4 * n_embd, bias=False)
    self.up_proj = nn.Linear(n_embd, 4 * n_embd, bias=False)
    self.down_proj = nn.Linear(4 * n_embd, n_embd, bias=False)
    self.act_fn = nn.GELU(approximate='tanh')
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    gate = self.gate(x)
    up = self.up_proj(x)
    gate = self.act_fn(gate)
    x = gate * up
    x = self.down_proj(x)
    x = self.dropout(x)

    return x



In [ ]:
class FeedForward(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embd,  4 * n_embd),
      # nn.ReLU(),
      nn.GELU(approximate='tanh'),
      nn.Linear(4 * n_embd, n_embd),
      nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)

In [ ]:
class Block(nn.Module):
  def __init__(self, n_embd, n_head):

    super().__init__()
    self.sa = MultiHeadAttention(n_head, head_size)
    # self.ffwd = FeedForward(n_embd)
    self.ffwd = GemmaFeedForward(n_embd)
    # self.ln1 = nn.LayerNorm(n_embd)
    # self.ln2 = nn.LayerNorm(n_embd)
    self.rms1 = nn.RMSNorm(n_embd)
    self.rms2 = nn.RMSNorm(n_embd)

  def forward(self, x):
        # x = x + self.sa(self.ln1(x))
        # x = x + self.ffwd(self.ln2(x))
        x = x + self.sa(self.rms1(x))
        x = x + self.ffwd(self.rms2(x))
        return x

In [ ]:
class GemmaLanguageModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()

    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
    self.final_norm = nn.RMSNorm(n_embd)
    self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
    self.dropout = nn.Dropout(dropout)

  def forward(self, idx, targets=None):

    B, T = idx.shape # Corrected from B, T, C = idx.shape

    tok_emb = self.token_embedding_table(idx)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device))
    # did this to match the gemma architecture
    tok_emb = tok_emb*(n_embd**0.5)
    x = tok_emb + pos_emb
    x = self.dropout(x)
    x = self.blocks(x)
    x = self.final_norm(x)
    logits = self.lm_head(x)

    if targets is None:
      loss = None
    else:
      # C here is vocab_size
      logits = logits.view(B*T, logits.shape[-1])
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
          # crop idx to the last block_size token
          idx_cond = idx[:, -block_size:]
          logits, loss = self(idx_cond)
          logits = logits[:, -1, :] # becomes (B, C)
          probs = F.softmax(logits, dim=-1) # (B, C)
          idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
          idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [ ]:
@torch.no_grad()
def estimate_loss(model_name):
    out = {}
    if model_name == 'gemma':
      model = gemma_model
    else:
      model = model
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)

            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


In [ ]:
# model = BigramLanguageModel(vocab_size=len(chars))
# m = model.to(device)
# # print the number of parameters in the model
# print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# # create a PyTorch optimizer
# optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


In [ ]:
gemma_model = GemmaLanguageModel(vocab_size=len(chars)).to(device)

print(sum(p.numel() for p in gemma_model.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(gemma_model.parameters(), lr=learning_rate, weight_decay=0.1)

14.308992 M parameters


In [ ]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss('gemma')
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = gemma_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


step 0: train loss 4.2562, val loss 4.2624
step 500: train loss 2.3932, val loss 2.4311
step 1000: train loss 2.2327, val loss 2.3215
step 1500: train loss 1.9559, val loss 2.0890
step 2000: train loss 1.7552, val loss 1.9173
step 2500: train loss 1.6357, val loss 1.8259
step 3000: train loss 1.5434, val loss 1.7588
step 3500: train loss 1.4764, val loss 1.7048


In [ ]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


In such content
Contence, I'ld talk the gaten sell for suster.

EDWARD:
Now, gracious Marcius, wenthy king, and the stoops
Prospsal the pride that I make leave, but fold
Shut were in born throne thee. I'll the block:
There avoid; betwixt him will hence,
For prevail as too.

BENVOLIO:
O God! if thou hearine to my coming, chosens,
Have done.

DERCAS:
Have you decersed, and that affoic day;
And you, mother, all this neither exclain,
Tighters on her long in the catterper;
Against ten the troals of t


## Comparision Matrix

**Baseline:** step 4999: train loss 1.1525, val loss 1.5088

### Output (Baseline)

In such content
Contence, I'ld talk the gaten sell for suster.

EDWARD:
Now, gracious Marcius, wenthy king, and the stoops
Prospsal the pride that I make leave, but fold
Shut were in born throne thee. I'll the block:
There avoid; betwixt him will hence,
For prevail as too.

BENVOLIO:
O God! if thou hearine to my coming, chosens,
Have done.

DERCAS:
Have you decersed, and that affoic day;
And you, mother, all this neither exclain,
Tighters on her long in the catterper;
Against ten the troals of t
